In [16]:
from transformers import TFAutoModel, AutoTokenizer
from datasets import load_dataset
import tensorflow as tf
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [17]:
model = TFAutoModel.from_pretrained("bert-base-uncased")

2025-08-16 19:47:25.176100: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 93763584 exceeds 10% of free system memory.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized fr

In [3]:
dataset = load_dataset("SetFit/emotion")
print(dataset)

Repo card metadata block was not found. Setting CardData to empty.
Generating test split: 100%|██████████| 2000/2000 [00:00<00:00, 383006.48 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 2000
    })
})


In [18]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch['text'], padding = True, truncation = True)

encoded_dataset = dataset.map(tokenize, batched = True, batch_size = None)

Map: 100%|██████████| 2000/2000 [00:00<00:00, 14614.43 examples/s]


In [20]:
trainDataset = encoded_dataset["train"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "token_type_ids"],
    label_cols=["label"],
    shuffle=True,
    batch_size=32
)

valDataset = encoded_dataset["validation"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "token_type_ids"],
    label_cols=["label"],
    shuffle=True,
    batch_size=32
)

testDataset = encoded_dataset["test"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "token_type_ids"],
    label_cols=["label"],
    shuffle=True,
    batch_size=32
)


In [21]:
class BERTForClassification(tf.keras.Model):
    def __init__(self, bert_model, num_classes):
        super().__init__()
        self.bert = bert_model
        self.classifier = tf.keras.layers.Dense(num_classes, activation='softmax')
        
    def call(self, inputs):
        bert_outputs = self.bert(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            token_type_ids=inputs.get("token_type_ids")
        )
        cls_output = bert_outputs.last_hidden_state[:, 0, :]  
        return self.classifier(cls_output)

In [22]:
textClassifier = BERTForClassification(model, num_classes=6)
textClassifier.compile(
	optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5), 
	loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
	metrics = ['accuracy']
)

history = textClassifier.fit(trainDataset, validation_data = valDataset, epochs=3)

Epoch 1/3


/home/devcontainers/miniconda3/envs/agents-env/lib/python3.11/site-packages/keras/src/backend/tensorflow/nn.py:708: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(
2025-08-16 19:48:10.525009: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator bert_for_classification_2_1/tf_bert_model_2/bert/embeddings/assert_less/Assert/Assert


500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.1668 - loss: 1.8641

2025-08-16 19:49:10.514035: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator bert_for_classification_2_1/tf_bert_model_2/bert/embeddings/assert_less/Assert/Assert
2025-08-16 19:49:19.068878: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator bert_for_classification_2_1/tf_bert_model_2/bert/embeddings/assert_less/Assert/Assert


500/500 ━━━━━━━━━━━━━━━━━━━━ 80s 127ms/step - accuracy: 0.1670 - loss: 1.8639 - val_accuracy: 0.3500 - val_loss: 1.6141
Epoch 2/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 58s 116ms/step - accuracy: 0.3637 - loss: 1.6004 - val_accuracy: 0.3895 - val_loss: 1.5472
Epoch 3/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 58s 117ms/step - accuracy: 0.4144 - loss: 1.5358 - val_accuracy: 0.4325 - val_loss: 1.5122


In [24]:
textClassifier.evaluate(testDataset)

63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 71ms/step - accuracy: 0.4407 - loss: 1.5047


[1.4807347059249878, 0.45899999141693115]